# Lab 1 — Prompt chaining + memory for a Loan Application (Azure OpenAI)

In this lab, you will build a loan-application assistant using Azure OpenAI that can chain prompts reliably across multiple steps and retain useful context (“memory”) without repeating questions. You will implement two complementary memory patterns: structured memory (extracting and storing key applicant details as clean JSON) and retrieval memory (using embeddings to store and recall relevant prior notes when needed). By the end, you will have a practical blueprint for designing multi-step LLM workflows for financial intake scenarios—covering data collection, validation, summarization, and decision-ready outputs—while keeping responses consistent, auditable, and secure.

## 0) Prerequisites
- Python 3.10+
- Packages: `openai`, `numpy`, `pandas` (optional)

If needed, install:


In [ ]:
!pip -q install openai numpy pandas

## 1) Configure Azure OpenAI connection
Set these environment variables **before** running:
- `AZURE_OPENAI_ENDPOINT` (example: `https://agenticaiengineer.openai.azure.com/`)
- `AZURE_OPENAI_API_KEY`
- (optional) `AZURE_OPENAI_API_VERSION` (default used below)


In [ ]:
import os
from openai import AzureOpenAI

AZURE_OPENAI_ENDPOINT = os.getenv(
    "AZURE_OPENAI_ENDPOINT", "https://agenticaiengineer.openai.azure.com/"
)
AZURE_OPENAI_API_KEY = os.environ.get(
    "AZURE_OPENAI_API_KEY",
    "<REPLACE_WITH_YOUR_AZURE_OPENAI_KEY>",
)  # required
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

if not AZURE_OPENAI_API_KEY:
    raise ValueError(
        "Missing AZURE_OPENAI_API_KEY env var. Set it securely (do not hardcode keys)."
    )

CHAT_DEPLOYMENT = "gpt-4o-mini"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"

client = AzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)

print("AzureOpenAI client ready")

## 2) Helper functions (chat + embeddings + cosine similarity)
We’ll build tiny wrappers so the lab is easy to follow.

In [ ]:
import json
import numpy as np
from typing import List, Dict, Any, Optional, Tuple


def chat(
    messages: List[Dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 600
) -> str:
    resp = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


def embed(texts: List[str]) -> np.ndarray:
    resp = client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=texts,
    )
    vectors = [d.embedding for d in resp.data]
    return np.array(vectors, dtype=np.float32)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12
    return float(np.dot(a, b) / denom)


print("Helpers loaded")

## 3) Define a simple loan policy (for consistent decisions)
This policy is intentionally small for training. In real production systems, use approved policy documents and human review.

In [12]:
LOAN_POLICY = """Loan policy (training):
- Collect: name, age, employment status, monthly income, monthly debt payments, loan amount, term (months), purpose.
- Basic eligibility:
  - age >= 18
  - stable income required (employed or self-employed with consistent income)
- Risk hints:
  - Debt-to-Income (DTI) = monthly_debt / monthly_income
  - DTI <= 0.35: low risk
  - 0.35 < DTI <= 0.50: medium risk
  - DTI > 0.50: high risk (manual review or reject)
- Always explain which inputs were used and what is missing.
"""

## 4) Prompt chaining design
We’ll chain prompts in **stages**:
1. Intake: ask for missing info
2. Normalize: convert to a clean JSON record
3. Compute: deterministic DTI + rules
4. Decision: approve / manual review / reject with explanation

We’ll also add **memory** so the system remembers key facts across turns.

In [13]:
SYSTEM_INTAKE = """You are a loan application assistant.
Be concise, ask only for missing fields.
Never invent personal data. If something is unknown, ask.
"""

SYSTEM_MEMORY_WRITER = """You extract structured memory from a conversation.
Output ONLY valid JSON with these keys:
- applicant: {full_name, age, employment, monthly_income, monthly_debt, loan_amount, term_months, purpose}
- missing_fields: [list of any still missing]
If unknown, set value to null.
"""

SYSTEM_DECISION = """You are a loan decision assistant using the provided policy and applicant record.
You must:
- cite which applicant fields you used
- compute DTI as monthly_debt / monthly_income (if possible)
- recommend one: APPROVE, MANUAL_REVIEW, or REJECT
- keep the tone professional
"""

## 5) Start a sample applicant conversation
We’ll simulate a multi-turn intake where the user provides info gradually.

In [ ]:
conversation: List[Dict[str, str]] = [
    {"role": "system", "content": SYSTEM_INTAKE},
    {
        "role": "user",
        "content": "Hi, I want to apply for a personal loan. My name is Riya Sharma.",
    },
]

assistant_msg = chat(conversation)
conversation.append({"role": "assistant", "content": assistant_msg})
print(assistant_msg)

## 6) Add the user’s follow-up details (turn 2)

In [ ]:
conversation.append(
    {
        "role": "user",
        "content": "I am 29, employed full-time. I earn about 120,000 THB per month and pay 25,000 THB as debts monthly. I need 300,000 THB for home renovation over 24 months.",
    }
)

assistant_msg = chat(conversation)
conversation.append({"role": "assistant", "content": assistant_msg})
print(assistant_msg)

## 7) Build **structured memory** (a compact JSON record)
Instead of keeping the full chat forever, we’ll periodically summarize key facts into JSON.

In [ ]:
memory_writer_messages = [
    {"role": "system", "content": SYSTEM_MEMORY_WRITER},
    {
        "role": "user",
        "content": "Conversation:"
        + json.dumps(conversation, ensure_ascii=False, indent=2),
    },
]

memory_json_text = chat(memory_writer_messages, temperature=0)
print(memory_json_text)

memory = json.loads(memory_json_text)
memory

## 8) Deterministic calculation: compute DTI and prepare an applicant record
We’ll keep business rules in Python so results are consistent.

In [ ]:
def compute_dti(
    monthly_income: Optional[float], monthly_debt: Optional[float]
) -> Optional[float]:
    if monthly_income is None or monthly_debt is None or monthly_income == 0:
        return None
    return float(monthly_debt) / float(monthly_income)


app = memory.get("applicant", {})
monthly_income = app.get("monthly_income")
monthly_debt = app.get("monthly_debt")

dti = compute_dti(monthly_income, monthly_debt)
dti

## 9) Generate a decision (stage 4) using policy + applicant record
Notice: we pass the **policy text** + **structured record** to the model.

In [ ]:
decision_messages = [
    {"role": "system", "content": SYSTEM_DECISION},
    {
        "role": "user",
        "content": f"""Policy:{LOAN_POLICY} Applicant record: {json.dumps(app, indent=2)}Computed DTI: {dti}""",
    },
]

decision = chat(decision_messages, temperature=0.2, max_tokens=450)
print(decision)

## 10) Embedding-based memory (retrieve the right past facts)
If the conversation becomes long, we can store **memory chunks** and retrieve the most relevant ones using embeddings.

In [ ]:
# A tiny in-memory "vector store"
vector_store: List[Dict[str, Any]] = []


def add_memory_chunk(text: str, meta: Dict[str, Any]):
    vec = embed([text])[0]
    vector_store.append({"text": text, "vec": vec, "meta": meta})


def retrieve(query: str, k: int = 3) -> List[Dict[str, Any]]:
    if not vector_store:
        return []
    qv = embed([query])[0]
    scored = []
    for item in vector_store:
        score = cosine_sim(qv, item["vec"])
        scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{"score": s, **it} for s, it in scored[:k]]


# Add a few chunks we might want later
add_memory_chunk(
    "Applicant: Riya Sharma, 29, employed full-time.", {"type": "identity"}
)
add_memory_chunk(
    "Income: 120000 THB/month. Debt payments: 25000 THB/month.", {"type": "financials"}
)
add_memory_chunk(
    "Loan request: 300000 THB for home renovation, term 24 months.", {"type": "request"}
)

print(f"Stored {len(vector_store)} memory chunks")

## 11) Test retrieval: ask a question later and pull relevant memory

In [ ]:
query = "What is the applicant's monthly income and debt payments?"
hits = retrieve(query, k=2)
for h in hits:
    print(f"score={h['score']:.4f}  meta={h['meta']}  text={h['text']}")

## 12) Use retrieved memory in a new response
We’ll simulate the applicant returning later with a follow-up question.

In [ ]:
followup_question = "Can you remind me what DTI you used and why the decision was made?"
context_hits = retrieve("DTI inputs income debt", k=3)

retrieved_context = "\n".join([f"- {h['text']}" for h in context_hits])

messages = [
    {
        "role": "system",
        "content": "You are a loan assistant. Answer using ONLY the retrieved memory + the computed DTI provided.",
    },
    {
        "role": "user",
        "content": f"Retrieved memory: {retrieved_context} Computed DTI: {dti} User question: {followup_question}",
    },
]

print(chat(messages, temperature=0.1, max_tokens=250))

## 13) Quick exercise (for learners)
1. Add a new conversation turn where the applicant changes their income.
2. Update structured memory.
3. Re-compute DTI and re-run the decision.
4. Add a memory chunk for the updated income and validate retrieval.
